In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-10c0d377-9ebc-f562-6928-7c9545777e83, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [2]:
GITHUB_USERNAME = "aaliinae"

assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME"
assert len(GITHUB_USERNAME.strip()) > 0

print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: aaliinae


In [13]:
!sed -i '1i import re' distributed_cuda_solver.py

In [14]:
!head -n 10 distributed_cuda_solver.py

import re
import os
import sys
import json
import math
import hashlib
import subprocess
import time
import re
import numpy as np


In [15]:
!mpirun --allow-run-as-root --oversubscribe -np 4 \
python3 distributed_cuda_solver.py aaliinae

[rank 0] coords=[0, 0] neighbors=(-2,2,-2,1)
[rank 0] local shape=(258, 258), seed=16854581959340721065
[rank 2] coords=[1, 0] neighbors=(0,-2,-2,3)
[rank 2] local shape=(258, 258), seed=16854581959340721065
[rank 1] coords=[0, 1] neighbors=(-2,3,0,-2)
[rank 1] local shape=(258, 258), seed=16854581959340721065
[rank 3] coords=[1, 1] neighbors=(1,-2,2,-2)
[rank 3] local shape=(258, 258), seed=16854581959340721065
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likel

# Analytical & Forensic Engineering Responses

## Question 1 — Memory Coalescing & Strided Column Packing

The GPU arrays use row-major (C-contiguous) memory layout. Therefore, adjacent columns are separated by a stride of 258 elements. Each element occupies 8 bytes (float64).

The top and bottom halo rows are contiguous in memory and can be transferred directly. However, the left and right halo columns are non-contiguous because consecutive elements are separated by 258 × 8 = 2064 bytes.

Each column contains 256 elements:

- Required bytes per column: 256 × 8 = 2048 bytes.
- Required bytes for both columns: 4096 bytes.

If each column were transferred as one contiguous bounding span, the span would contain:

(255 × 258) + 1 = 65,791 elements.

The unnecessary data transferred for both columns would be:

2 × (65,791 − 256) × 8 = 1,048,560 bytes.

Therefore, dedicated GPU packing kernels significantly reduce unnecessary PCIe traffic by extracting only the required column elements into contiguous device buffers.

## Question 2 — Shared-Memory Tile Indexing & Bank Conflict Analysis

The CUDA kernel uses an 18 × 18 shared-memory tile to support a 16 × 16 output region with a one-cell halo apron.

For thread (tx=0, ty=0), the center element is:

- Center: smem[1,1]
- North: smem[0,1]
- South: smem[2,1]
- West: smem[1,0]
- East: smem[1,2]

For thread (tx=15, ty=15):

- Center: smem[16,16]
- North: smem[15,16]
- South: smem[17,16]
- West: smem[16,15]
- East: smem[16,17]

On NVIDIA Compute Capability 7.5, shared memory contains 32 banks with a 4-byte bank width.

Since each float64 element occupies 8 bytes, it spans two 32-bit banks.

For consecutive 64-bit accesses along a row, two-way bank conflicts can occur because 32 threads access 64-bit values across only 32 banks.

For column accesses with an 18-element row stride, the stride corresponds to 36 banks. Since 36 modulo 32 equals 4, the accesses map repeatedly to a smaller subset of banks, potentially producing four-way conflicts for a warp accessing consecutive rows of the same column.

Thus, shared-memory bank conflicts depend on the access pattern and the 18-element tile stride.

## Question 3 — PTX Assembly Audit & Warp Synchronization

The PTX audit produced the following results:

| PTX instruction | Count |
|---|---:|
| st.shared.f64 | 4 |
| ld.shared.f64 | 10 |
| bar.sync | 1 |

The non-zero instruction counts demonstrate that the compiled kernel uses shared-memory stores, shared-memory loads, and thread-block synchronization.

The 16 × 16 threadblock contains 256 threads, corresponding to 8 warps.

If cuda.syncthreads() is omitted, some warps may begin stencil calculations before other warps have completed loading the shared-memory tile.

This creates a read-after-write hazard: a warp may read an apron or neighboring cell before another warp has written the corresponding shared-memory value.

The synchronization barrier ensures that all threads finish loading the tile before stencil calculations begin.

## Question 4 — Multi-Process Context Sharing on a Single Physical GPU

The experiment used four independent MPI processes sharing one NVIDIA Tesla T4 GPU.

The measured execution times were:

| Execution mode | Time |
|---|---:|
| Serialized | 5035.19 ms |
| Overlapped | 2763.89 ms |

The speedup was:

Speedup = 5035.19 / 2763.89 ≈ 1.82×

The overlapped implementation reduced execution time by approximately 45.1%.

Each MPI process has its own CUDA context. The NVIDIA driver schedules work from these contexts onto the GPU's streaming multiprocessors (SMs). GPU execution and DMA resources are shared between processes.

In the overlapped implementation, boundary data is packed on the GPU and transferred to pinned host memory. Non-blocking MPI communication is then initiated while the interior stencil computation executes on a separate CUDA stream.

This allows communication and computation to overlap.

However, all four processes share the same physical GPU and its DMA resources. Therefore, the overlap is subject to resource contention and cannot achieve the same scaling as four dedicated GPUs connected through InfiniBand.

## Question 5 — Point-to-Point Message Deadlock & MPI.PROC_NULL Semantics

Replacing non-blocking MPI communication with blocking MPI_Send followed by MPI_Recv can introduce deadlocks.

For sufficiently large messages, MPI implementations may use a rendezvous protocol rather than eagerly buffering the entire message.

In a blocking communication pattern, a sender may wait for the receiver to initiate the matching receive operation.

If neighboring ranks all execute blocking sends in the same directional order, every process may wait for another process to receive its message.

This creates a circular wait condition and can cause deadlock.

Non-blocking MPI_Isend and MPI_Irecv avoid this problem by allowing communication requests to be posted before waiting for completion.

MPI.PROC_NULL simplifies physical boundary handling. Communication with MPI.PROC_NULL completes without transferring data.

For example, Rank (0,0) has no upper or left neighbor. Its corresponding ghost boundaries can therefore be filled with zero values, implementing the required homogeneous Dirichlet boundary condition without additional special communication logic.

## Question 6 — Explicit CFL Stability & Energy Conservation Proof

The measured global maximum diffusivity was:

κmax = 0.23999962497115201

The grid spacing was:

Δx = 1/513 ≈ 0.00194931774

The explicit diffusion stability limit is:

Δtmax = Δx² / (4 × κmax)

Substituting the experimental values:

Δtmax ≈ 3.95817 × 10⁻⁶

The simulation used:

Δt = 1.0 × 10⁻⁶

Therefore:

Δt / Δtmax ≈ 0.2526

The selected time step is below the calculated diffusion stability limit.

The global thermal energy obtained from MPI Allreduce was:

| Step | Global energy | Global L2 norm |
|---:|---:|---:|
| 0 | 0.04207134875 | 0.12791284166 |
| 100 | 0.04206560653 | 0.12716516705 |
| 200 | 0.04206168885 | 0.12643302397 |
| 300 | 0.04205803408 | 0.12571583747 |
| 400 | 0.04205443660 | 0.12501309479 |

Both thermal energy and the L2 norm decrease monotonically throughout the simulation.

The final energy is lower than the initial energy, demonstrating dissipative behavior over the 400 time steps.

The serialized and overlapped implementations also produced identical final states, with a maximum absolute difference of:

parity_max_diff = 0.0

This confirms numerical agreement between the two execution modes for this experiment.